In [ ]:
import os
import cv2

origin_path = r"D:\Documents\Projects\WoWsBot\training\dataset\images_warship"
output_path = r"D:\Documents\Projects\WoWsBot\training\dataset\images_compass2"

for file in os.listdir(origin_path):
    image_path = os.path.join(origin_path, file)
    # print(image_path)
    if file.endswith(".png"):
        pass
    elif file.endswith(".jpg"):
        pass
    else:
        continue
    
    img = cv2.imread(image_path)
    if img is None:
        print("图片加载失败")
        continue
        
    h, w = img.shape[:2]
    roi = img[h-230:h, :230]  # 裁剪左下角
    print(roi.shape)
    output = os.path.join(output_path, file)
    print(output)
    cv2.imwrite(output, roi)

In [ ]:
import os
import cv2
import numpy as np
from typing import List, Tuple


def mse(img1: np.ndarray, img2: np.ndarray) -> float:
    """
    计算两个图像之间的均方误差(MSE)
    
    Args:
        img1: 第一个图像
        img2: 第二个图像
        
    Returns:
        两个图像之间的MSE值，值越小表示越相似
    """
    return float(np.mean((img1 - img2) ** 2))


def load_image(image_path: str) -> np.ndarray:
    """
    加载图像并调整大小以便比较
    
    Args:
        image_path: 图像文件路径
        
    Returns:
        加载的图像
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"无法加载图像: {image_path}")
    
    # 调整图像大小以确保可以比较
    img = cv2.resize(img, (230, 230))
    return img


def find_similar_images(dir1: str, dir2: str, threshold: float = 100.0) -> List[Tuple[str, str, float]]:
    """
    查找两个目录中的相似图像
    
    Args:
        dir1: 第一个图像目录
        dir2: 第二个图像目录
        threshold: 相似度阈值，MSE小于此值的图像对被认为是相似的
        
    Returns:
        相似图像对的列表，每个元素包含(图像1路径, 图像2路径, MSE值)
    """
    similar_pairs = []
    
    # 获取两个目录中的所有图像文件
    files1 = [f for f in os.listdir(dir1) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    files2 = [f for f in os.listdir(dir2) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    print(f"目录1 ({dir1}) 包含 {len(files1)} 个图像文件")
    print(f"目录2 ({dir2}) 包含 {len(files2)} 个图像文件")
    
    total_comparisons = len(files1) * len(files2)
    print(f"开始进行 {total_comparisons} 次比较...")
    
    count = 0
    for file1 in files1:
        img1_path = os.path.join(dir1, file1)
        try:
            img1 = load_image(img1_path)
        except ValueError as e:
            print(e)
            continue
            
        for file2 in files2:
            img2_path = os.path.join(dir2, file2)
            try:
                img2 = load_image(img2_path)
            except ValueError as e:
                print(e)
                continue
                
            # 计算MSE
            mse_value = mse(img1, img2)
            
            # 如果MSE低于阈值，则认为是相似图像
            if mse_value < threshold:
                similar_pairs.append((img1_path, img2_path, mse_value))
                print(f"发现相似图像: {file1} 和 {file2}, MSE: {mse_value:.2f}")
                
            count += 1
            if count % 1000 == 0:
                print(f"已完成 {count}/{total_comparisons} 次比较")
    
    # 按MSE值排序，最相似的图像对排在前面
    similar_pairs.sort(key=lambda x: x[2])
    
    return similar_pairs


def main():
    """
    主函数
    """
    # 定义目录路径
    imgs1 = r"D:\Documents\Projects\WoWsBot\training\dataset\images_compass2"
    imgs2 = r"D:\Documents\Projects\WoWsBot\training\dataset\images_compass2"
    
    # 检查目录是否存在
    if not os.path.exists(imgs1):
        print(f"目录不存在: {imgs1}")
        return
        
    if not os.path.exists(imgs2):
        print(f"目录不存在: {imgs2}")
        return
    
    # 查找相似图像，使用较低的阈值来查找极度相似的图像
    similar_pairs = find_similar_images(imgs1, imgs2, threshold=50.0)
    
    # 输出结果
    print("\n" + "="*50)
    print("查找完成!")
    print("="*50)
    
    if similar_pairs:
        print(f"找到 {len(similar_pairs)} 对相似图像:")
        for i, (path1, path2, mse_value) in enumerate(similar_pairs):
            if path1 == path2:
                continue
            print(f"{i+1}. {os.path.basename(path1)} <-> {os.path.basename(path2)} (MSE: {mse_value:.2f})")
            if mse_value < 30.0:
                print(f"删除相似图像...")
                num1 = path1.split("_")[1].split(".")[0]
                num2 = path2.split("_")[1].split(".")[0]
                if num1 > num2:
                    # os.remove(path1)
                elif num1 < num2:
                    # os.remove(path2)
                # os.remove(path2)
    else:
        print("未找到相似图像")


if __name__ == "__main__":
    main()

In [ ]:
import os
from PIL import Image
dir = r"D:\Documents\Projects\WoWsBot\training\dataset\images_warship"

format = "png"
for file in os.listdir(dir):
    image_path = os.path.join(dir, file)
    # print(image_path)
    if file.endswith(".png"):
        continue
    elif file.endswith(".jpg"):
        pass
    else:
        continue

    image = Image.open(image_path)
    image.save(os.path.join(dir, file[:-4] + "." + format), format)
